# Kaggle GPU Runner — Faster R-CNN & Controlled Ablations (F0 → F5)
Chạy lần lượt các cell bên dưới trên Kaggle GPU T4 (bật **Internet** + thêm Secret `WANDB_API_KEY`).

In [ ]:
%cd /kaggle/working
!git clone -b minh https://github.com/dohoangminh-vn/AIT3001.Group6.traffic.sign.detection.git btl
%cd /kaggle/working/btl
!pip install -q pycocotools sahi albumentations pyyaml
!python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('W&B API key loaded from Kaggle Secrets.')
except Exception as e:
    print('Fallback W&B offline mode:', e)

In [ ]:
# Chỉnh đường dẫn tới dataset Zalo AI trên /kaggle/input/...
RAW_DIR = '/kaggle/input/za-traffic-2020/za_traffic_2020/traffic_train'
!python scripts/02_prepare_data.py data.raw_dir={RAW_DIR}
!pytest -q

In [ ]:
# Chạy chuỗi Ablation có kiểm soát F0 -> F5 (cùng seed=42, cùng split_street_id.json)
IMG_DIR = f'{RAW_DIR}/images'
!python scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=default roi_size=7 loss_type=ce scheduler=cosine out_dir=runs/frcnn/f0_default
!python scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=small_sign_ar roi_size=7 loss_type=ce scheduler=cosine out_dir=runs/frcnn/f1_small_ar
!python scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=small_sign_ar roi_size=7 loss_type=ce scheduler=plateau out_dir=runs/frcnn/f2_plateau
!python scripts/04_train_frcnn.py img_dir={IMG_DIR} out_dir=runs/frcnn/f3_focal_roi14
!python scripts/04_train_frcnn.py img_dir={IMG_DIR} tiled=true copy_paste=true min_size=800 max_size=800 out_dir=runs/frcnn/f5_tiled640

In [ ]:
# Nếu phiên Kaggle bị ngắt, nối tiếp từ last.pth bằng --resume:
# !python scripts/04_train_frcnn.py img_dir={IMG_DIR} out_dir=runs/frcnn/f3_focal_roi14 --resume runs/frcnn/f3_focal_roi14/last.pth

# Suy luận Batch-4 SAHI (F4 trên trọng số F3, và F5 + SAHI)
!python scripts/06_infer_sahi_demo.py img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth out_dir=runs/frcnn/f3_focal_roi14
!python scripts/06_infer_sahi_demo.py img_dir={IMG_DIR} weights=runs/frcnn/f5_tiled640/best.pth out_dir=runs/frcnn/f5_tiled640

In [ ]:
# Chấm điểm tổng hợp pycocotools + Ma trận nhầm lẫn + FPN/Grad-CAM + Video Demo FPS
!python scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds \
  F0_Default=outputs/preds/f0_default_val.json \
  F1_SmallAR=outputs/preds/f1_small_ar_val.json \
  F2_Plateau=outputs/preds/f2_plateau_val.json \
  F3_CBFocal_RoI14=outputs/preds/f3_focal_roi14_val.json \
  F4_SAHI=outputs/preds/f3_focal_roi14_sahi_val.json \
  F5_Tiled_SAHI=outputs/preds/f5_tiled640_sahi_val.json

!python scripts/07_analyze_and_demo.py --mode features img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
!python scripts/07_analyze_and_demo.py --mode video img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
!zip -r /kaggle/working/frcnn_results.zip outputs/eval outputs/preds runs/frcnn/*/resolved_config.yaml